# Data creation

The survey ground truth and the model matrix. The first half of this stage is **R and Quarto** and
does not run from a notebook — it reads DHS/MICS microdata with `haven`. Those files live in
`src/data_creation/` and keep their own numbering; this notebook runs the Python half and checks
what the R half produced.

| step | file | language |
| --- | --- | --- |
| 00 | `src/data_creation/00_data_cleaning_primary.R` | R — unzips and files survey archives (`DRY_RUN`) |
| 01 | `src/data_creation/01_population_data.R` | R — UN WPP counts joined to the ground truth |
| 02 | `src/data_creation/02_ground_truth_data_calculation.qmd` | Quarto — internet/mobile penetration from DHS, MICS, ITU |
| 03 | `src/data_creation/03_internet_indicator_cleaning.qmd` | Quarto — writes `internet_mobile_indicator_clean.csv` |
| 04 | `src/data_creation/04_data_availability_check.R` | R — coverage check |

```bash
Rscript src/run_qmd.R src/data_creation/02_ground_truth_data_calculation.qmd
```

The UN population panel is Python and is rebuilt here.

In [1]:
import _bootstrap  # noqa: F401  — puts src/ on sys.path
import pandas as pd

import params
import population

STAGE = 'data_creation'
print('UN edition in use :', params.UN_POP_EDITION)
print('population panel  :', params.UN_POP_PROCESSED.name)

UN edition in use : shipped
population panel  : un_1950_2023_processed.csv


## 1. UN population

`src/population.py` rebuilds the 1950–2023 age/sex panel from the raw WPP workbooks. It takes
~15 minutes and writes a date-stamped file, so it is left off by default — set `REBUILD = True`
to run it. Adopting a new edition is `params.UN_POP_EDITION`, not a file copy (D35).

In [2]:
REBUILD = False

if REBUILD:
    population.main()

pop = pd.read_csv(params.UN_POP_PROCESSED, usecols=['iso3', 'Year', '18_inf_t'])
print(f'{len(pop):,} rows | {pop.iso3.nunique()} countries | {pop.Year.min()}-{pop.Year.max()}')
pop.head()

17,775 rows | 237 countries | 1950-2024


,iso3,Year,18_inf_t
0,ABW,1950,22703.0
1,ABW,1951,23038.0
2,ABW,1952,23415.0
3,ABW,1953,23812.0
4,ABW,1954,24255.0


## 2. What the R chain produced

The harmonised survey outcomes. `pop_year` records which population year each survey was matched
to — the shared rule is exact year, else the latest earlier one (D32).

In [3]:
matches = sorted((params.EXTERNAL / 'national/adolescent_modelling')
                 .glob('update_full_groundtruth_*.csv'))
gt = pd.read_csv(matches[-1])
print(f'{matches[-1].name}: {len(gt)} country-years')
print('by survey type:', gt.survey_type.value_counts().to_dict())
print('missing population:', int(gt.pop_15_to_49_female.isna().sum()))
print('population carried forward:', int((gt.pop_year != gt.year).sum()))
gt.head()

update_full_groundtruth_20260812.csv: 90 country-years
by survey type: {'mics6': 36, 'dhs7': 29, 'dhs8': 21, 'continuous dhs8': 3, 'continuous dhs7': 1}
missing population: 0
population carried forward: 1


,iso3,country,year,survey_type,internet_men,internet_wom,mobile_men,mobile_wom,internet_fm_ratio,mobile_fm_ratio,pop_15_to_49_female,pop_15_to_49_male,pop_year
0,SEN,Senegal,2017,continuous dhs7,0.450408,0.289751,0.831702,0.679760,0.643309,0.817312,3651800.0,3622675,2017
1,SEN,Senegal,2018,continuous dhs8,0.544060,0.393988,0.833277,0.707980,0.724163,0.849633,3775470.0,3754777,2018
2,SEN,Senegal,2019,continuous dhs8,0.582990,0.467399,0.845071,0.697247,0.801727,0.825075,3903860.0,3892922,2019
3,AGO,Angola,2015,dhs7,0.373536,0.174995,0.703222,0.512360,0.468482,0.728589,6357994.0,6167890,2015
4,ARM,Armenia,2015,dhs7,0.889742,0.853496,0.987729,0.967414,0.959262,0.979433,784071.0,678734,2015


## 3. The modelling outcomes and the model matrix

`src/outcomes.py` merges the indicators into the modelling outcomes; `src/predictors.py` and
`src/predictors_by_year.py` build the year-aligned matrices; `src/missingness.py` drives the
imputation exclusion lists. They are top-level scripts rather than importable steps, so they are
run from the shell — see `doc/workflow.md`.

What matters downstream is the fitting panel they end at:

In [4]:
for indicator in params.FINAL_MODEL['indicators']:
    path = (params.PROCESSED / 'combined_data/updated_ground_truth_and_fb'
            / indicator / params.FINAL_MODEL['dataset'])
    panel = pd.read_csv(path)
    itu = panel[f'{indicator}_survey_type'].astype(str).str.contains('itu', case=False).sum()
    print(f'{indicator:9s} {len(panel):4d} rows  ({itu} ITU)  {panel.iso3.nunique()} countries')

internet   108 rows  (37 ITU)  95 countries
mobile      99 rows  (24 ITU)  83 countries


## Notes

- The panel is named `..._itu_deleted.csv` but **contains** ITU rows — the name records a removed
  file, not removed observations (D37).
- Model fitting is `02_model_fitting/02_01_fit_final_models.ipynb`.